## 第 1 周挑战 — 宣传册生成器（泰卢固语 → 英语翻译）

## 练习目标（理念）

本 notebook 在 Ed Donner 第 5 天「宣传册生成器」基础上扩展：先抓泰卢固语新闻站，生成泰卢固语宣传册，再**第二次 LLM 调用**译成英语——演示把多次 API 调用串成流水线（pipeline）。

### 功能
- 用 BeautifulSoup / 自带 `scraper` 抓取网站链接与正文
- 第一次 LLM 调用：生成泰卢固语公司宣传册
- 第二次 LLM 调用：把宣传册译成英语
- 展示「一步一事」的 prompt 拆分：生成与翻译职责分离

### 和本课 Day 1 / Day 5 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user messages | 链接筛选、写宣传册、翻译各有一套 prompt |
| JSON 结构化输出 | `response_format={"type": "json_object"}` |
| 流式输出 `stream=True` | 最终版 `create_brochure` 边生成边拼字符串 |
| 多步流水线 | 抓取 → 筛链接 → 写册 → 翻译 |

### 怎么跑

1. 从上到下依次运行单元格（Shift+Enter）
2. `.env` 里准备好 `OPENAI_API_KEY`；确保 `scraper` 模块可用
3. 示例站点是 Eenadu（`https://www.eenadu.net/`），可改成别的公司站


In [ ]:
# ========== 导入：后面抓网页、调 API、展示 Markdown 都靠这些 ==========

# 若导入失败：请确认已在激活的 (llms) 虚拟环境里运行本 notebook
# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：把模型返回的 JSON 字符串解析成 Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里渲染 / 更新 Markdown
from IPython.display import Markdown, display, update_display
# 从本目录 scraper 模块导入：抓页面链接列表、抓页面正文内容
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 初始化：加载密钥、做一次粗检查、定模型与客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# 从环境变量取出 OpenAI API Key（名字必须是 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：有值、以 sk-proj- 开头、长度够——只是启发式，不能保证密钥一定有效
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    # 失败提示字符串保持英文（原样），便于对照官方排查 notebook
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 本练习里「筛链接」等步骤默认用的模型名；字符串必须和你账号可用模型一致
MODEL = 'gpt-5-nano'
# 创建 OpenAI 客户端；默认会从环境变量读 OPENAI_API_KEY
openai = OpenAI()


In [ ]:
# ========== 试抓：先看一眼首页上能扫到哪些链接 ==========

# 调用 scraper：抓取 Eenadu 首页上的链接列表（返回值通常是 URL 字符串列表）
links = fetch_website_links("https://www.eenadu.net/")
# 在笔记本里直接写出变量名：最后一行表达式会被显示出来，方便目检
links


## 第一步：让模型判断哪些链接相关

首页上往往有几十上百个链接。下一步用 **system + user prompt** 让模型挑出写宣传册真正有用的页面（About、团队、栏目、联系等），并要求以 **JSON** 返回，方便后面程序解析。


In [ ]:
# ========== system prompt：告诉模型「怎么筛链接、用什么 JSON 格式回答」==========
# 发给模型的指令字符串保持英文（可运行 / 影响行为的 prompt 不翻译）
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the news company,
such as links to an About page, Editorial Team page, News Categories/Sections, 
Contact page, or Advertising/Reach pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "editorial team page", "url": "https://another.full.url/team"}
    ]
}
"""


In [ ]:
# ========== 组装 user prompt：把「该站全部链接」塞进给模型的用户消息 ==========

def get_links_user_prompt(url):
    # 先写任务说明（英文 prompt 保留）；{url} 会插进当前要分析的网站
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    # 再次抓取该站链接列表（与上面试抓同一套 scraper）
    links = fetch_website_links(url)
    # 把每条链接用换行拼进 prompt，模型才能「看见」候选集
    user_prompt += "\n".join(links)
    # 返回完整 user 消息字符串，稍后放进 messages
    return user_prompt


In [ ]:
# ========== 预览：打印即将发给模型的 user prompt（调试用）==========
# 看一眼拼好的长文本，确认链接列表真的进去了
print(get_links_user_prompt("https://www.eenadu.net/"))


In [ ]:
# ========== 调用 API：让模型筛相关链接，并解析为 Python 字典 ==========

def select_relevant_links(url):
    # Chat Completions：system 定规则，user 给具体链接列表
    response = openai.chat.completions.create(
        # 使用前面定义的 MODEL 常量
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            # user 内容由 get_links_user_prompt 现场拼出
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        # 要求返回 JSON 对象（不是自由文本），便于 json.loads
        response_format={"type": "json_object"}
    )
    # 取出助手消息的文本内容（应是 JSON 字符串）
    result = response.choices[0].message.content
    # 解析成字典，例如 {"links": [{"type": ..., "url": ...}, ...]}
    links = json.loads(result)
    return links


In [ ]:
# ========== 试跑：对 Eenadu 首页做一次「相关链接」筛选 ==========
select_relevant_links("https://www.eenadu.net/")


In [ ]:
# ========== 同名函数增强版：加打印，方便观察进度（会覆盖上一格定义）==========

def select_relevant_links(url):
    # 开始调用前打印：哪个 URL、哪个模型
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    # 打印筛出了多少条，便于快速确认 JSON 结构里有 'links' 键
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
# ========== 再试跑一次：确认带日志的版本工作正常 ==========
select_relevant_links("https://www.eenadu.net/")


## 第二步：制作宣传册！

把落地页正文 + 相关子页正文组装成**另一段 user prompt**，再交给模型写泰卢固语宣传册。注意：后面还有「翻译」第二步，这里只负责「写册」。


In [ ]:
# ========== 聚合内容：落地页 + 各相关链接页的正文，拼成一大段 Markdown 文本 ==========

def fetch_page_and_all_relevant_links(url):
    # 抓首页 / 落地页正文
    contents = fetch_website_contents(url)
    # 调用上一格的 select_relevant_links：LLM 帮你挑该读哪些子页
    relevant_links = select_relevant_links(url)
    # 用 Markdown 标题把「落地页」和「相关链接」分区，便于模型阅读
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    # 逐个相关链接抓正文；某页失败就跳过，不让整条流水线崩掉
    for link in relevant_links['links']:
        try:
            # link['type'] 是模型标注的页面类型（about / contact 等）
            result += f"\n\n### Link: {link['type']}\n"
            # 按模型返回的完整 URL 再抓一页
            result += fetch_website_contents(link["url"])
        except Exception as e:
            # 打印跳过原因后 continue
            print(f"Skipping {link['url']}: {e}")
            continue
    return result


In [ ]:
# ========== 预览聚合结果：打印抓到的长文本（可能很慢、很长）==========
print(fetch_page_and_all_relevant_links("https://www.eenadu.net/"))


In [ ]:
# ========== brochure 的 system prompt：规定「用泰卢固语写短宣传册」==========
# prompt 正文保持英文；要求输出 markdown、不要包在代码块里
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages 
from a Telugu news company website and creates a short brochure about the 
company in Telugu language for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""


In [ ]:
# ========== 组装写册用的 user prompt：公司名 + 抓到的页面内容（截断防超长）==========

def get_brochure_user_prompt(company_name, url):
    # 任务说明保持英文；插入公司名
    user_prompt = f"""
You are looking at a Telugu news company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in Telugu language in markdown without code blocks.\n\n
"""
    # 把落地页+相关页正文追加进 prompt
    user_prompt += fetch_page_and_all_relevant_links(url)
    # 截断到前 5000 字符：控制 token / 费用；可能丢掉后面页面内容
    user_prompt = user_prompt[:5_000]
    return user_prompt


In [ ]:
# ========== 试组装：看写册 prompt 长什么样（会触发抓取+筛链接）==========
get_brochure_user_prompt("Eenadu", "https://www.eenadu.net/")


In [ ]:
# ========== 第一版 create_brochure：非流式，一次拿回泰卢固语宣传册并展示 ==========

def create_brochure(company_name, url):
    # 注意：这里模型写死为 gpt-4.1-mini（与前面筛链接用的 MODEL 常量不同）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    # 取出完整助手回复（泰卢固语宣传册 Markdown）
    telugu_brochure = response.choices[0].message.content
    # 在笔记本里用 Markdown 漂亮渲染
    display(Markdown(telugu_brochure))


In [ ]:
# ========== 试生成：对 Eenadu 跑一遍「只写泰卢固语宣传册」==========
create_brochure("Eenadu", "https://eenadu.net/")


## 第三步：为什么要单独做「翻译」prompt？

与其让模型**一步直接用英语写宣传册**，这里拆成两次 LLM 调用：

1. **第一次调用** — 根据网站内容生成泰卢固语宣传册  
2. **第二次调用** — 把泰卢固语宣传册翻译成英语  

每个 prompt 只做一件事：
- `brochure_system_prompt` → 专心写好宣传册  
- `translation_system_prompt` → 专心做准确翻译  

这样代码更模块化、可复用——翻译函数以后也能拿去译别的内容，不限于宣传册。


In [ ]:
# ========== 翻译用的 system prompt：泰卢固语 → 英语，保持版式与语气 ==========
# 指令字符串保持英文（影响翻译行为，不翻译 prompt 本身）
translation_system_prompt = """
You are an expert translator fluent in Telugu and English.
You will be given a company brochure written in Telugu.
Translate the entire brochure accurately and naturally into English.
Maintain the same formatting, structure and tone as the original.
Respond in markdown without code blocks.
"""


In [ ]:
# ========== 组装翻译用的 user prompt：把泰卢固语正文嵌进去 ==========

def get_translation_user_prompt(telugu_brochure):
    # f-string：说明任务 + 贴上待译全文；prompt 英文保留
    user_prompt = f"""Here is a company brochure written in Telugu.
Translate it accurately and naturally into English.

{telugu_brochure}
"""
    return user_prompt


In [ ]:
# ========== 最终版 create_brochure：流式写泰卢固语册 → 再流式译成英语（覆盖上一版）==========

def create_brochure(company_name, url):
    # ----- 第一次 API：流式生成泰卢固语宣传册 -----
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        # stream=True：边生成边返回增量 delta，不必等整段结束
        stream=True
    )
    # 累加容器：把每个 chunk 的文本拼成完整宣传册
    telugu_brochure = ""
    for chunk in response:
        # delta.content 可能为 None（例如结束标记），用 or "" 避免 TypeError
        telugu_brochure += chunk.choices[0].delta.content or ""
    # 先展示泰卢固语完整结果
    display(Markdown(telugu_brochure))

    # ----- 第二次 API：流式把泰卢固语宣传册译成英语 -----
    translation_response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": translation_system_prompt},
            {"role": "user", "content": get_translation_user_prompt(telugu_brochure)}
        ],
        stream=True
    )
    english_brochure = ""
    for chunk in translation_response:
        english_brochure += chunk.choices[0].delta.content or ""
    # 再展示英语宣传册
    display(Markdown(english_brochure))


In [ ]:
# ========== 端到端试跑：写册 + 翻译两条流水线一次走完 ==========
create_brochure("Eenadu", "https://eenadu.net/")
